# Question 2 (verbatim instruction)

"For the above selected companies, using csv files to track their weekly average stock prices, weekly average volume, weekly average market shares, and (the following may only have quarterly reports) revenues, costs, EBI/EVITDA, Net Income, EPS, P/E."

## What this notebook delivers

This notebook tracks the 30 selected companies on a weekly basis for average stock prices, average volume, and average market shares. It tracks revenues, costs, EBI/EVITDA, Net Income, EPS, and P/E on a quarterly basis. It outputs weekly_tracking.csv and quarterly_tracking.csv. It allows Notebobook 3 to analyze patterns.

## Inputs

### From Notebook 1
- **`data/processed/q1_us_tech_with_shares.csv`** — full cleaned universe (~1,000 rows)
- **`data/processed/q1_selected_30_all_tech.csv`** — 30 selected companies

### From Bloomberg Terminal
- **`data/raw/Price_daily.csv`** — Daily Prices and volumes for each ticker
- **`data/raw/Quarterly_Fundamentals.csv`** — Data found using the BDS function. Fields used: SALES_REV_TURN, EBITDA, NET_INCOME, IS_EPS. This csv has report dates
- **`data/raw/Quarterly_Fundamentals_BDH.csv`** — Data found using the BDH function. Fields used: SALES_REV_TURN, EBITDA, NET_INCOME, IS_EPS. This csv has quarterly dates
- **`data/raw/Shares_Outstanding_History.csv`** — A table for each ticker's Shares Outstanding

## Outputs

- **`data/processed/weekly_tracking.csv`** — Weekly tracking for average stock prices, average volume, and average market shares
- **`data/processed/quarterly_tracking.csv`** — Quarterly tracking for revenues, costs, EBI/EVITDA, Net Income, EPS, and P/E
- **`data/processed/quarterly_tracking_changes.csv`** — A collapsed verson of **quarterly_tracking.csv**. See caveats

- Note: **pe_w_from_eps** is **NaN** for **eps_q <= 0**

## Caveats:

- Fundamentals are a panel not a snapshot. They are built from **Quarterly_Fundamentals_BDH.csv** and joined to weeks with **merge_asof(drection="backward")** on the effective date. No week sees a quarter not yet reported. Pre-first-report weeks are dropped
- **quarterly_tracking.csv** has one row per (ticker, BDH observation date). ~7 quarters x 30 tickers
- **pe_w_from_eps = price_avg / (eps_q * 4)** and is **NaN** when **eps_q <= 0**. The 4 annualization assumes **IS_EPS** is a quarterly diluted figure
- **Shares_Outstanding_History.csv** are quarterly. **Weekly market_cap_w = price_avg * as-of-reported** shares is a hybrid, not a point-in-time cap. Intra-quarter buybacks/dilutions are invisible. Missing quarters (DAIC, HCTI, WOLF for 2024-06 and 2024-09) are forward-filled from 2024-03 and flagged via **shares_forward_filled**
- Denominator 2 is the static snapshot sum of **Market Cap:Q** from Notebook 1. Only the numerator moves weekly. This is a hybrid index. Recon abs diff vs snapshot reaches a few pp. The assert tolderance is 5.0 pp
- Volume in **Price_daily.csv** is sometimes repeated across consecutive dates for some tickers. **volume_avg** is a mean over **possibly_duplicated** values and is not a clean weekly average
- **volume_avg** can be **NaN** for weeks where price exists but volume doesn't
- AVGO and KLAC share bases in **Shares_Outstanding_History.csv** are not on the same split basis as **Price_daily.csv**. Absolute dollar market cap is off by the split factor which propagates into **share_of_selected_30_pct_w** and **share_of_static_universe_pct_w** for all tickers. They are excluded from both numerators and denominators. Their share columns are **NaN** and **split_basis_suspect=True** flags them
- Costs are proxied as Revenue - EBITDA (**rev_less_ebitda_q**). This is not EBIT and not EBIT-less-D&A. True operating expense is not derivable
- **quarterly_tracking.csv** is one row per (ticker, date). Not per fiscal quarter. The BDH file is daily-indexed and carries no fiscal period label. Row count is ~13.6k not 30. **quarterly_tracking_changes.csv** collapses contiguous runs of identical fundamentals to their first day, giving one row per report-date change with ~452 rows. It's derived from **quarterly_tracking.csv** and shouldn't be used as the source of truth
- **volume_avg** is a mean over de-duplicated days only. Days where volume equals the previous day's volume for the same ticker are excluded from the weekly mean (flagged in **prices["volume_duplicated"]**). NVDA final-week **volume_avg** was ~2x the Nasdaq spot volume before the fix


Loading CSV Files

In [11]:
from pathlib import Path
import pandas as pd
import numpy as np
import math

def find_repo_root(start: Path) -> Path:
    p = start.resolve()
    for _ in range(10):
        if (p / "data" / "raw").is_dir():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise FileNotFoundError(f"data/raw/ not found above {start}")

try:
    NOTEBOOK_DIR = Path(__vsc_ipynb_file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd()

REPO_ROOT = find_repo_root(NOTEBOOK_DIR)
DATA_RAW = REPO_ROOT / "data" / "raw"
DATA_PROCESSED = REPO_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents = True, exist_ok = True)

PRICES_DAILY = DATA_RAW / "Price_daily.csv"
SHARES_QTR = DATA_RAW / "Shares_Outstanding_History.csv"
FUNDAMENTALS_SNAP = DATA_RAW / "Quarterly_Fundamentals.csv"
FUNDAMENTALS_BDH = DATA_RAW / "Quarterly_Fundamentals_BDH.csv"
SELECT_30 = DATA_PROCESSED / "q1_selected_30_all_tech.csv"
OUT_WEEKLY = DATA_PROCESSED / "weekly_tracking.csv"
OUT_QUARTERLY = DATA_PROCESSED / "quarterly_tracking.csv"

NEED_30 = ["bucket", "ticker", "Name", "industry", "Market Cap:Q",
           "EPS:Q", "NI / Profit:Q", "EBITDA:Q", "Revenue:Q", "share_of_all_tech_pct"]

if not SELECT_30.is_file():
    raise FileNotFoundError(f"Missing {SELECT_30}. Run 01 first")

sel_30 = pd.read_csv(SELECT_30, dtype={"ticker": str})
missing = set(NEED_30) - set(sel_30.columns)
if missing:
    raise ValueError(f"q1_selected_30_all_tech.csv missing columns: {missing}")

sel_30 = sel_30[NEED_30].copy()
sel_30["ticker"] = sel_30["ticker"].str.upper().str.strip()
assert sel_30["ticker"].is_unique and len(sel_30) == 30

TICK_30 = sel_30["ticker"].tolist()

Load Daily Prices

In [12]:
def parse_prices_transposed(path: Path) -> pd.DataFrame:
    raw = pd.read_csv(path, header=None, dtype=str, sep="\t", engine="python")
    raw = raw.dropna(how="all").reset_index(drop=True)

    date_row_idx, dates_col = None, None
    for i in range(min(len(raw), 10)):
        for j in range(min(3, raw.shape[1])):
            cell = str(raw.iloc[i, j]).strip()
            if cell == "Dates":
                date_row_idx, dates_col = i, j + 1
                break
        if date_row_idx is not None:
            break

    if date_row_idx is None:
        raise ValueError(f"No 'Dates cell in first 3 cols of first 10 rows")

    dates = pd.to_datetime(raw.iloc[date_row_idx, dates_col:], errors="coerce")
    
    body = raw.iloc[date_row_idx + 1:, :].reset_index(drop=True)
    if body.shape[1] < 2:
        raise ValueError(f"Price_daily.csv parsed to {body.shape[1]} column(s). Expected at least 2")

    records = []
    ticker = None
    field = None

    for i in range(len(body)):
        c0 = body.iloc[i, 0]
        c1 = body.iloc[i, 1]
        if pd.notna(c0) and str(c0).strip() not in ("", "nan"):
            ticker = str(c0).replace(" US Equity", "").strip().upper()
        if pd.notna(c1) and str(c1).strip() in ("Last Price", "Volume"):
            field = str(c1).strip()
            vals = body.iloc[i, dates_col:].replace("#N/A N/A", np.nan)
            vals = pd.to_numeric(vals, errors="coerce")
            for d, v in zip(dates, vals):
                records.append({"ticker": ticker, "date": d, "field": field, "value": v})

    long = pd.DataFrame(records)
    long = long.dropna(subset=["date"])
    wide = long.pivot_table(index=["ticker", "date"], columns="field", values="value",
                            aggfunc="first").reset_index()
    wide = wide.rename(columns={"Last Price": "close", "Volume": "volume"})
    wide = wide.dropna(subset=["close"])
    wide = wide[wide["close"] > 0]

    # Guard for nonsensical values
    wide["med"] = wide.groupby("ticker")["close"].transform("median")
    wide = wide[wide["close"] <= 1000 * wide["med"]].drop(columns="med")

    return wide.sort_values(["ticker", "date"]).reset_index(drop = True)

prices = parse_prices_transposed(PRICES_DAILY)

_p = prices.sort_values(["ticker", "date"]).copy()
_p["prev_vol"] = _p.groupby("ticker")["volume"].shift()
_p["volume_duplicated"] = _p["volume"].eq(_p["prev_vol"]) & _p["volume"].notna()
prices = prices.merge(_p[["ticker", "date", "volume_duplicated"]], on=["ticker", "date"], how="left")
prices["volume_duplicated"] = prices["volume_duplicated"].fillna(False)

_dup_frac = (prices.groupby("ticker")["volume_duplicated"].mean().sort_values(ascending=False).head(10))

print("Fraction of days with volume identical to previous day (Top 10):")
print(_dup_frac.to_string())

missing_t = set(TICK_30) - set(prices["ticker"])
assert not missing_t, f"no prices for: {sorted(missing_t)}"
print(f"{len(prices)} daily rows across {prices['ticker'].nunique()} tickers")



Fraction of days with volume identical to previous day (Top 10):
ticker
DAIC    0.186813
HCTI    0.136264
YYAI    0.129670
BKYI    0.120879
AMD     0.120879
AVPT    0.120879
FRSH    0.120879
FSLY    0.120879
SOUN    0.120879
SMTK    0.120879
13390 daily rows across 30 tickers


Weekly Resampling

In [13]:
def to_weekly(prices: pd.DataFrame) -> pd.DataFrame:

    p = prices.copy()
    p["date"] = pd.to_datetime(p["date"])
    p = p.set_index("date").sort_index()

    # Price mean over all days
    price_w = (p.groupby("ticker").resample("W-FRI", include_groups=False)
              .agg(price_avg=("close", "mean"))
              .reset_index().rename(columns={"date": "week_end"})
    )

    # Volume mean over non-duplicated days
    vol_clean = p.loc[~p["volume_duplicated"]]
    vol_w = (
        vol_clean.groupby("ticker").resample("W-FRI", include_groups=False)
        .agg(volume_avg=("volume", "mean")).reset_index().rename(columns={"date": "week_end"})
    )

    weekly = price_w.merge(vol_w, on=["ticker", "week_end"], how="left")
    weekly = weekly.dropna(subset=["price_avg"])
    weekly["week_end"] = pd.to_datetime(weekly["week_end"])
    
    return weekly

weekly = to_weekly(prices)
assert weekly["week_end"].dt.day_of_week.eq(4).all()
assert not weekly.duplicated(["ticker", "week_end"]).any()

Shares Outstanding

In [14]:
def parse_shares_transposed(path: Path) -> pd.DataFrame:

    raw = pd.read_csv(path, header=None, dtype=str, sep=",", engine="python")
    raw = raw.dropna(how="all").reset_index(drop=True)
    
    dates_idx = raw.index[raw.iloc[:, 0].astype(str).str.strip().eq("Dates")]
    if len(dates_idx) == 0:
        raise ValueError(f"No 'Dates' row in {path.name}")

    dates_idx = int(dates_idx[0])

    tickers_row = raw.iloc[dates_idx - 1, :].astype(str).str.strip().tolist()
    labels_row = raw.iloc[dates_idx, :].astype(str).str.strip().tolist()

    data = raw.iloc[dates_idx + 1:, :].reset_index(drop=True)

    recs = []

    for i in range(1, raw.shape[1]):

        ticker_cell = tickers_row[i]
        label_cell = labels_row[i]
        if not ticker_cell or ticker_cell == "nan":
            continue
        if label_cell != "Shares Outstanding":
            continue
        ticker = ticker_cell.replace(" US Equity", "").strip().upper()

        for j in range(len(data)):
            date_cell = str(data.iloc[j, 0]).strip()
            val_cell = str(data.iloc[j, i]).strip()

            d = pd.to_datetime(date_cell, errors="coerce")
            if pd.isna(d):
                continue
            if val_cell in ("", "#N/A N/A", "nan"):
                continue

            v = pd.to_numeric(val_cell, errors="coerce")
            if pd.isna(v) or v <= 0:
                continue
            recs.append({"ticker": ticker, "date": d, "shares_out": v})

    s = pd.DataFrame(recs)
    if s.empty: return s
    s = (s.sort_values(["ticker", "date"]).drop_duplicates(["ticker", "date"], keep="last")
         .reset_index(drop=True))

    return s

shares = parse_shares_transposed(SHARES_QTR)
assert len(shares) > 0, "shares parsed empty - check the field label string"

all_tickers = sorted(shares["ticker"].unique())
all_quarters = sorted(shares["date"].unique())
expected_n = len(all_tickers) * len(all_quarters)
present_n = len(shares)
print(f"shares: {present_n} rows, {len(all_tickers)} tickers x {len(all_quarters)} quarters = {expected_n} expected {expected_n - present_n} missing")

if present_n < expected_n:
    full = pd.MultiIndex.from_product([all_tickers, all_quarters], names=["ticker", "date"])

    have = pd.MultiIndex.from_frame(shares[["ticker", "date"]])
    missing = full.difference(have).to_frame(index=False)
    print("\nMissing (ticker, quarter) pairs:")
    print(missing.to_string(index=False))

shares: 204 rows, 30 tickers x 7 quarters = 210 expected 6 missing

Missing (ticker, quarter) pairs:
ticker       date
  DAIC 2024-06-28
  DAIC 2024-09-30
  HCTI 2024-06-28
  HCTI 2024-09-30
  WOLF 2024-06-28
  WOLF 2024-09-30


Weekly Market Cap and Share Denominators

In [15]:
def add_weekly_shares(weekly: pd.DataFrame, shares: pd.DataFrame,
            TICK_30: list[str]) -> pd.DataFrame:

    w = weekly.sort_values(["week_end", "ticker"]).copy()
    s = shares.rename(columns={"date": "shares_date"}).sort_values(["shares_date", "ticker"])

    w = pd.merge_asof(w, s, left_on="week_end", right_on="shares_date", by="ticker", direction="backward", allow_exact_matches=True)
    w["shares_forward_filled"] = w["shares_date"] < w["week_end"]
    w["shares_age_days"] = (w["week_end"] - w["shares_date"]).dt.days

    snap_mcap_series = sel_30.set_index("ticker")["Market Cap:Q"]

    probe = (
        w.sort_values("week_end").groupby("ticker").tail(1)
        .assign(snap_mcap=lambda d: d["ticker"].map(snap_mcap_series))
        .dropna(subset=["snap_mcap"])
    )
    probe["ratio"] = probe["snap_mcap"] / (probe["price_avg"] * probe["shares_out"])

    log_ratio = np.log10(probe["ratio"].replace([0, np.inf, -np.inf], np.nan).dropna())
    scale_exp = int(round(log_ratio.median()))
    SHARES_SCALE = 10 ** scale_exp

    print(f"Detected shares units scale: 10^{scale_exp} (multiply shares_out by {SHARES_SCALE:g})")
    print(f"  median implied scale: {10**log_ratio.median():.3g}")
    print(f"  per-ticker implied scale (top 5 by |log10|):")
    _probe_diag = probe.assign(
        implied_scale=probe["ratio"],
        log_dev=(np.log10(probe["ratio"]) - log_ratio.median()).abs(),
    ).sort_values("log_dev", ascending=False).head(5)
    print(_probe_diag[["ticker", "price_avg", "shares_out", "snap_mcap", "implied_scale", "log_dev"]].to_string(index=False))

    # Sanity: most tickers must agree with the chosen power of 10
    within = ((np.log10(probe["ratio"]) - log_ratio.median()).abs() < 0.5).mean()
    assert within > 0.8, (
        f"Only {within:.0%} of tickers agree on the shares units scale 10^{scale_exp}. "
        "Shares file may mix units across tickers — investigate before proceeding."
    )

    w["shares_out"] = w["shares_out"] * SHARES_SCALE
    
    w["market_cap_w"] = w["price_avg"] * w["shares_out"]

    # Stale-shares summary
    stale = (
        w.loc[w["shares_forward_filled"]].groupby("ticker").agg(n_weeks=("week_end", "nunique"), max_age_days=("shares_age_days", "max"))
        .sort_values("max_age_days", ascending=False)
    )
    if not stale.empty:
        print("forward-filled shares (Stale Market Cap) by ticker:")
        print(stale.to_string())
    if not stale.empty:
        assert stale["max_age_days"].max() < 180, (
            f"Shares forward-filled for {int(stale['max_age_days'].max())} days. Shares data too stale to trust for market cap"
        )

    # Split-basis detection
    snap_ref = (snap_Universe[["ticker", "Market Cap:Q", "Price:M-3"]]
                .rename(columns={"Market Cap:Q": "snap_mcap", "Price:M-3": "snap_price"})
                .dropna(subset=["snap_mcap", "snap_price"])
                .assign(implied_snapshot_shares=lambda d: d["snap_mcap"] / d["snap_price"]).set_index("ticker"))

    last_week = w.sort_values("week_end").groupby("ticker").tail(1).copy()
    last_week["snap_price"] = (last_week["ticker"].map(snap_ref["snap_price"]))
    last_week["implied_snapshot_shares"] = (last_week["ticker"].map(snap_ref["implied_snapshot_shares"]))
    last_week["shares_ratio"] = (last_week["shares_out"] / last_week["implied_snapshot_shares"])

    suspects = last_week.loc[
        (last_week["shares_ratio"] < 0.5) | (last_week["shares_ratio"] > 2.0),
        ["ticker", "snap_price", "price_avg", "shares_out", "implied_snapshot_shares", "shares_ratio"]
    ].sort_values("shares_ratio")

    print("Split-basis suspects (shares_out / implied_snapshot_shares far from 1.0):")
    print(suspects.to_string(index=False))

    SPLIT_SUSPECT_TICKERS = set(suspects["ticker"])

    ALLOWED_SPLIT_SUSPECTS = {"AVGO", "KLAC", "BTLN", "YYAI"}
    ALLOWED_STALE_SHARES = {"WOLF"}

    unexpected = SPLIT_SUSPECT_TICKERS - (ALLOWED_SPLIT_SUSPECTS | ALLOWED_STALE_SHARES)
    assert not unexpected, (
        f"Unexpected split basis suspects beyond the allow-list:"
        f"{sorted(unexpected)}. Inspect the printed table. Add each"
        f"to ALLOWED_SPLIT_SUSPECTS only after verifying that its shares_ratio"
        f"is a clean multiple and that the mismatch is in the shares file, not the price file"
        f"Or it may need to go to ALLOWED_STALE_SHARES only if its stale-shares row shows missing quarters"
    )
    
    w["split_basis_suspect"] = w["ticker"].isin(ALLOWED_SPLIT_SUSPECTS | ALLOWED_STALE_SHARES)
    w["share_exclusion_reason"] = np.select(
        [w["ticker"].isin(ALLOWED_SPLIT_SUSPECTS), w["ticker"].isin(ALLOWED_STALE_SHARES)],
        ["split_basis", "stale_shares"], default=None
    )

    clean = w[~w["split_basis_suspect"]]

    # Denominator 1: Sum of Selected 30 tickers' market caps that week
    denom_30 = clean.groupby("week_end")["market_cap_w"].sum().rename("denom_30")
    w = w.merge(denom_30, on="week_end", how="left")

    # Denominator 2: Full US-tech universe
    denom_full = snap_Universe.loc[~snap_Universe["ticker"].isin(SPLIT_SUSPECT_TICKERS), "Market Cap:Q"].sum()
    w["denom_full"] = denom_full

    ok = ~w["split_basis_suspect"]
    w.loc[ok, "share_of_selected_30_pct_w"] = w.loc[ok, "market_cap_w"] / w.loc[ok, "denom_30"] * 100
    w.loc[ok, "share_of_static_universe_pct_w"] = w.loc[ok, "market_cap_w"] / denom_full * 100
    w.loc[~ok, "share_of_selected_30_pct_w"] = np.nan
    w.loc[~ok, "share_of_static_universe_pct_w"] = np.nan

    return w

snap_Universe = pd.read_csv(DATA_PROCESSED / "q1_us_tech_with_shares.csv", dtype={"ticker": str})
snap_Universe["Market Cap:Q"] = pd.to_numeric(snap_Universe["Market Cap:Q"], errors="coerce")
snap_Universe = snap_Universe.dropna(subset=["Market Cap:Q"])
snap_Universe["Market Cap:Q"] = snap_Universe["Market Cap:Q"].astype(float)

weekly_mc = add_weekly_shares(weekly, shares, TICK_30)

Detected shares units scale: 10^6 (multiply shares_out by 1e+06)
  median implied scale: 1.13e+06
  per-ticker implied scale (top 5 by |log10|):
ticker  price_avg  shares_out  snap_mcap  implied_scale  log_dev
  YYAI  655.04000      0.0527   126427.2   3.662369e+03 2.490318
  SMTK    0.27760      0.5137  6292247.5   4.412419e+07 1.590597
  BTLN    6.27056      2.6346  2088239.0   1.264035e+05 0.952321
  HCTI    5.14000      0.1014  3670171.5   7.041826e+06 0.793606
  TRNR    1.54000      0.0439   206910.3   3.060532e+06 0.431717
forward-filled shares (Stale Market Cap) by ticker:
        n_weeks  max_age_days
ticker                       
AAPL         82          91.0
AGYS         82          91.0
AIP          82          91.0
AMAT         82          91.0
AMD          82          91.0
AVGO         82          91.0
AVPT         82          91.0
BKYI         82          91.0
BRZE         82          91.0
BTLN         82          91.0
FRSH         82          91.0
FSLY         82        

Parse Fundamentals BDH

In [16]:
def parse_bdh(path: Path) -> pd.DataFrame:

    raw = pd.read_csv(path, header=None, dtype=str, engine="python")
    raw = raw.dropna(how="all").reset_index(drop=True)

    dates_idx = None
    for i in range(min(len(raw), 10)):
        if str(raw.iloc[i, 0]).strip() == "Dates":
            dates_idx = i
            break
    if dates_idx is None:
        raise ValueError(f"No 'Dates row in {path.name}")

    ticker_row = raw.iloc[dates_idx - 1].tolist()
    field_row = raw.iloc[dates_idx].tolist()
    body = raw.iloc[dates_idx + 1:].reset_index(drop=True)

    blocks = []
    cur_ticker, cur_fields = None, []
    for j in range(1, raw.shape[1]):
        t = str(ticker_row[j]).strip()
        if t and t.lower() != "nan":
            if cur_ticker is not None:
                blocks.append((cur_ticker, cur_fields))
            cur_ticker, cur_fields = t.replace(" US Equity", "").strip().upper(), []
        
        f = str(field_row[j]).strip()
        if f in ("Revenue", "EBITDA", "Net Income/Net Profit (Losses)", "Basic Earnings per Share"):
            cur_fields.append((f, j))

    if cur_ticker is not None:
        blocks.append((cur_ticker, cur_fields))

    recs = []
    for ticker, fields in blocks:
        for i in range(len(body)):
            d = pd.to_datetime(str(body.iloc[i, 0]).strip(), errors="coerce")
            if pd.isna(d):
                continue
            for fname, col in fields:
                v = str(body.iloc[i, col]).strip()
                if v in ("", "nan", "#N/A N/A"):
                    continue
                val = pd.to_numeric(v, errors="coerce")
                if pd.isna(val):
                    continue
                recs.append({"ticker": ticker, "date": d, "field": fname, "value": val})

        long = pd.DataFrame(recs)
        if long.empty:
            raise ValueError("parse_bdh produced 0 rows. Layout mismatch")
    
    return (long.drop_duplicates(["ticker", "date", "field"], keep="last")
            .sort_values(["ticker", "date", "field"]).reset_index(drop=True))

fund_long = parse_bdh(FUNDAMENTALS_BDH)
fund_long = fund_long[fund_long["ticker"].isin(TICK_30)]

assert fund_long["ticker"].nunique() == 30, (
    f"parse_bdh yielded {fund_long['ticker'].nunique()} tickers. Parser broken"
    f"Present: {sorted(fund_long['ticker'].unique())}"
)

print(f"fund_long rows: {len(fund_long)} | tickers: {fund_long["ticker"].nunique()}")

fund_wide = (fund_long.pivot_table(index=["ticker", "date"], columns="field", values="value", aggfunc="first")
             .reset_index().rename(columns={"Revenue": "revenue_q", "EBITDA": "ebitda_q",
                                            "Net Income/Net Profit (Losses)": "ni_q",
                                            "Basic Earnings per Share": "eps_q"}))

fund_wide = fund_wide.sort_values(["ticker", "date"]).reset_index(drop=True)
print(f"fund_wide rows: {len(fund_wide)} | cols: {list(fund_wide.columns)}")

fund_long rows: 54340 | tickers: 30
fund_wide rows: 13650 | cols: ['ticker', 'date', 'eps_q', 'ebitda_q', 'ni_q', 'revenue_q']


Quarterly Fundamentals

In [17]:
def build_quarterly(sel_30: pd.DataFrame, fund_wide: pd.DataFrame) -> pd.DataFrame:

    # 'Costs" is found by Revenue - EBITDA

    fw = fund_wide.copy()
    fw["ticker"] = (fw["ticker"].astype(str).str.replace(r"\s+US\s+Equity$", "", regex=True).str.strip().str.upper())

    q = fw[fw["ticker"].isin(sel_30["ticker"])].copy()

    q = q.rename(columns={"Revenue": "revenue_q", "EBITDA": "ebitda_q",
                          "Net Income/Net Profit (Losses)": "ni_q",
                          "Basic Earnings per Share": "eps_q"})

    # Numeric Coercion
    for c in ["revenue_q", "ebitda_q", "ni_q", "eps_q"]:
        q[c] = pd.to_numeric(q[c], errors="coerce")

    q["rev_less_ebitda_q"] = q["revenue_q"] - q["ebitda_q"]

    if "effective_date" in q.columns:
        q_key = "effective_date"
    elif "date" in q.columns:
        q = q.rename(columns={"date": "effective_date"})
        q_key = "effective_date"
    else:
        raise KeyError("fund_wide has neither 'effective_date' nor 'date'")

    q = q.dropna(subset=[q_key]).sort_values(["ticker", q_key]).reset_index(drop=True)

    n_present = q["ticker"].nunique()
    assert n_present >= 25, (
        f"build_quarterly only sees {n_present}/30 tickers. parse_bdh is broken"
        f"Not a genuine data gap. Present: {sorted(q['ticker'].unique())}"
    )

    missing = sorted(set(sel_30["ticker"]) - set(q["ticker"].unique()))
    if missing:
        sentinel = q["effective_date"].max()
        filler = pd.DataFrame({"ticker": missing, "effective_date": [sentinel] * len(missing),
                               "revenue_q": np.nan, "ebitda_q": np.nan, 
                               "rev_less_ebitda_q": np.nan, "ni_q": np.nan, "eps_q": np.nan})
        q = pd.concat([q, filler], ignore_index=True)
        print(f"Filled {len(missing)} missing tickers with NaN rows: {missing}")

    q = q[["effective_date", "ticker", "revenue_q", "ebitda_q", "rev_less_ebitda_q", "ni_q", "eps_q"]]

    q = q.sort_values(["effective_date", "ticker"]).reset_index(drop=True)

    return q

quarterly = build_quarterly(sel_30, fund_wide)
print(f"Quarterly rows: {len(quarterly)}")
print(f"Quarterly columns: {sorted(quarterly.columns)}")
print(f"Tickers: {quarterly["ticker"].nunique()}")

Quarterly rows: 13650
Quarterly columns: ['ebitda_q', 'effective_date', 'eps_q', 'ni_q', 'rev_less_ebitda_q', 'revenue_q', 'ticker']
Tickers: 30


Fundamentals and Weekly P/E

In [18]:
def add_weekly_fundamentals(weekly_mc: pd.DataFrame, q_panel: pd.DataFrame) -> pd.DataFrame:

    w = weekly_mc.sort_values(["week_end", "ticker"]).reset_index(drop=True)
    q = q_panel.sort_values(["effective_date", "ticker"]).reset_index(drop=True)

    q_cols = ["revenue_q", "ebitda_q", "ni_q", "eps_q", "rev_less_ebitda_q"]

    missing = [c for c in q_cols if c not in q.columns]
    assert not missing, f"q_panel missing columns: {missing}"

    w = pd.merge_asof(w, q[["ticker", "effective_date", *q_cols]],
                      left_on="week_end", right_on="effective_date",
                      by="ticker", direction="backward", allow_exact_matches=True)
    w = w.rename(columns={"effective_date": "fund_asof_date"})

    # Weekly P/E: annualized quarterly EPS. NaN when EPS <= 0
    with np.errstate(divide="ignore", invalid="ignore"):
        w["pe_w_from_eps"] = (w["price_avg"] / (w["eps_q"] * 4.0)).where(w["eps_q"] > 0)


    return w.reset_index(drop=True)

weekly_full = add_weekly_fundamentals(weekly_mc, quarterly)

# Look Ahead Check
bad = weekly_full.loc[weekly_full["fund_asof_date"].notna() & (weekly_full["fund_asof_date"] > weekly_full["week_end"]),
                      ["ticker", "week_end", "fund_asof_date"]]
assert bad.empty, f"Look ahead rows after merge_asof:\n{bad.head(10)}"

assert (weekly_full["fund_asof_date"].dropna() <= 
        weekly_full.loc[weekly_full["fund_asof_date"].notna(), 
                        "week_end"]).all(), "Look Ahead: A week is using a quarter reported after that week"

print(f"weekly_full rows: {len(weekly_full)} | pe_w_from_eps non-NaN: {int(weekly_full["pe_w_from_eps"].notna().sum())}")


weekly_full rows: 2678 | pe_w_from_eps non-NaN: 1314


Final Weekly Table and Export

In [19]:
weekly_tracking = (
    
    weekly_full[[
        "ticker", "week_end", "price_avg", "volume_avg", "shares_out", "shares_forward_filled", 
        "shares_age_days", "market_cap_w", "split_basis_suspect", "share_exclusion_reason", "share_of_selected_30_pct_w", "share_of_static_universe_pct_w", 
        "revenue_q", "ebitda_q", "ni_q", "eps_q", "rev_less_ebitda_q", "pe_w_from_eps", "fund_asof_date"
    ]].sort_values(["ticker", "week_end"]).reset_index(drop=True)
                   
)

quarterly = quarterly[["effective_date", "ticker", "revenue_q", "ebitda_q",
                       "rev_less_ebitda_q", "ni_q", "eps_q"]]

quarterly = quarterly.sort_values(["ticker", "effective_date"]).reset_index(drop=True)

def collapse_to_changes(q: pd.DataFrame) -> pd.DataFrame:

    val_cols = ["revenue_q", "ebitda_q", "rev_less_ebitda_q", "ni_q", "eps_q"]
    q = q.sort_values(["ticker", "effective_date"]).reset_index(drop=True)

    changed = (q.groupby("ticker")[val_cols].apply(lambda g: g.ne(g.shift()).any(axis=1)).reset_index(level=0, drop=True))

    first = ~q.duplicated("ticker")

    return q.loc[changed | first].reset_index(drop=True)

quarterly_changes = collapse_to_changes(quarterly)

weekly_tracking.to_csv(OUT_WEEKLY, index=False)
quarterly.to_csv(OUT_QUARTERLY, index=False)
quarterly_changes.to_csv(OUT_QUARTERLY.with_name("quarterly_tracking_changes.csv"), index=False)

print(f"Weekly: {len(weekly_tracking):,} rows > {OUT_WEEKLY}")
print(f"Quarterly: {len(quarterly):,} rows > {OUT_QUARTERLY}")
print(f"Quarterly (Changes): {len(quarterly_changes):,} rows > {OUT_QUARTERLY.with_name('quarterly_tracking_changes.csv')}")
print(f"Week range: {weekly_tracking['week_end'].min().date()} > "
      f"{weekly_tracking['week_end'].max().date()}")

Weekly: 2,678 rows > C:\Users\mtwei\Documents\VSCode\Data Analysis Research\Data-Analysis-Research\USTechMarketShare\data\processed\weekly_tracking.csv
Quarterly: 13,650 rows > C:\Users\mtwei\Documents\VSCode\Data Analysis Research\Data-Analysis-Research\USTechMarketShare\data\processed\quarterly_tracking.csv
Quarterly (Changes): 452 rows > C:\Users\mtwei\Documents\VSCode\Data Analysis Research\Data-Analysis-Research\USTechMarketShare\data\processed\quarterly_tracking_changes.csv
Week range: 2024-05-03 > 2026-01-23


Reconciliation Check

In [20]:
# Last_week share_of_static_universe vs the static snapshot value
# Expected diff < 0.5
last_week = weekly_tracking.sort_values("week_end").groupby("ticker").tail(1).loc[lambda d: ~d["split_basis_suspect"]].copy()

recon = (
    last_week[["ticker", "share_of_static_universe_pct_w"]]
    .merge(sel_30[["ticker", "share_of_all_tech_pct"]], on="ticker")
    .assign(diff=lambda d: d["share_of_static_universe_pct_w"] - d["share_of_all_tech_pct"])
)

print("Recon diff (28 clean tickers, last week vs snapshot):")
print(recon["diff"].abs().describe())

mx = recon["diff"].abs().max()
assert mx < 5.0, f"static-denominator recon drifted {mx:.1f} pp (limit 5.0)"

snap_pe = snap_Universe.set_index("ticker")["P/E:Q"]
last_week = last_week.assign(pe_q_snapshot=last_week["ticker"].map(snap_pe))

# Tickers with weekly P/E > 100x
high_pe = (
    last_week[last_week["pe_q_snapshot"] > 100]
    [["ticker", "pe_q_snapshot", "pe_w_from_eps", "eps_q", "ni_q"]].sort_values("pe_q_snapshot", ascending=False)
)

print("\nSnapshot P/E > 100x (Last week cross check only):")
print(high_pe.to_string(index=False))

Recon diff (28 clean tickers, last week vs snapshot):
count    2.500000e+01
mean     3.396326e-01
std      8.299376e-01
min      6.037084e-07
25%      1.364729e-05
50%      4.249052e-03
75%      1.183137e-01
max      3.469844e+00
Name: diff, dtype: float64

Snapshot P/E > 100x (Last week cross check only):
ticker  pe_q_snapshot  pe_w_from_eps  eps_q   ni_q
  INTC         148.07      26.095238   0.63 3320.0
   AMD         144.82      90.142857   1.40 2297.0
